# Experiment 5.5 — Non-SNN history-conditioned WHAT experts

This notebook is **analysis-only**. It reads finalized Exp5.5 CSV/JSON artifacts and never trains models, launches Slurm jobs, or regenerates missing runs.

Primary question: does history-conditioned mapping outperform shared WHAT, absolute elapsed time, and matched-capacity reset-GRU gating?

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'scripts').is_dir():
            return candidate
    raise FileNotFoundError('repository root not found')

ROOT = find_repo_root() / 'notebooks' / 'artifacts' / 'experiment_5_5_non_snn_history_experts' / 'non_snn_history_experts_v1'
REQUIRED = [
    'references.csv', 'runs.csv', 'summary.csv', 'paired_deltas.csv',
    'state_diagnostics.csv', 'state_profiles.csv', 'q_only_probes.csv',
    'shuffled_q.csv', 'same_what_pairs.csv', 'manifest.json',
]
missing = [name for name in REQUIRED if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f'Exp5.5 is not finalized; missing artifacts: {missing}')
ROOT

## Locked protocol and aggregate metrics

In [ ]:
manifest = json.loads((ROOT / 'manifest.json').read_text(encoding='utf-8'))
runs = pd.read_csv(ROOT / 'runs.csv')
summary = pd.read_csv(ROOT / 'summary.csv')
references = pd.read_csv(ROOT / 'references.csv')
paired = pd.read_csv(ROOT / 'paired_deltas.csv')
display(pd.DataFrame([manifest]))
display(summary.sort_values('mean_test_balanced_accuracy', ascending=False))
display(references)

In [ ]:
condition_order = ['shared', 'clock', 'oracle_progress', 'reset_gru', 'ordered_gru']
fig, ax = plt.subplots(figsize=(9, 5))
for condition in condition_order:
    group = runs[runs['condition'] == condition].sort_values('seed')
    ax.plot(group['seed'].astype(str), group['test_balanced_accuracy'], marker='o', label=condition)
ax.set_xlabel('Experiment seed')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Exp5.5 paired test BA by condition')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## Paired mechanism contrasts

The most important contrast is `ordered_minus_reset`, which isolates recurrent history while holding GRU/state-head/expert capacity fixed.

In [ ]:
paired_summary = (paired.groupby('comparison')['delta_test_balanced_accuracy']
                  .agg(['mean', 'std', 'min', 'max', 'count'])
                  .sort_values('mean', ascending=False))
display(paired_summary)

fig, ax = plt.subplots(figsize=(10, 5))
for comparison, group in paired.groupby('comparison'):
    ax.plot(group['seed'].astype(str), group['delta_test_balanced_accuracy'], marker='o', label=comparison)
ax.axhline(0.0, linewidth=1)
ax.set_xlabel('Experiment seed')
ax.set_ylabel('Paired delta test BA')
ax.set_title('History/context mechanism contrasts')
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
plt.show()

## State occupancy, entropy, persistence, and clock/progress alignment

In [ ]:
state_diag = pd.read_csv(ROOT / 'state_diagnostics.csv')
profiles = pd.read_csv(ROOT / 'state_profiles.csv')
display(state_diag[state_diag['condition'].isin(['reset_gru', 'ordered_gru'])].head(30))

ordered_profile = profiles[(profiles['condition'] == 'ordered_gru') & (profiles['axis'] == 'relative_progress')]
profile_mean = ordered_profile.groupby(['position', 'state'], as_index=False)['probability'].mean()
fig, ax = plt.subplots(figsize=(10, 5))
for state, group in profile_mean.groupby('state'):
    ax.plot(group['position'], group['probability'], marker='o', label=f'state {int(state)}')
ax.set_xlabel('Relative progress (diagnostic axis only)')
ax.set_ylabel('Mean q probability')
ax.set_title('Ordered-GRU latent context vs relative progress')
ax.legend(ncol=2, fontsize=8)
ax.grid(alpha=0.25)
plt.show()

## q-only class information

High q-only BA near the full model would indicate that q may be collapsing toward a class code instead of serving mainly as context.

In [ ]:
q_probe = pd.read_csv(ROOT / 'q_only_probes.csv')
q_test = q_probe[q_probe['split'] == 'test']
display(q_test.groupby(['condition', 'feature'])['balanced_accuracy'].agg(['mean', 'std', 'min', 'max']))

## Ordered-GRU temporal-alignment ablation

In [ ]:
shuffled = pd.read_csv(ROOT / 'shuffled_q.csv')
display(shuffled.groupby('seed')['balanced_accuracy'].agg(['mean', 'std', 'min', 'max']))
display(paired[paired['comparison'] == 'ordered_minus_shuffled_q'])

## Same-WHAT / different-history diagnostic

Rows compare highly similar nonzero frozen-WHAT vectors from different samples and classes, then report latent-context distance and local class-support changes.

In [ ]:
same_what = pd.read_csv(ROOT / 'same_what_pairs.csv')
if same_what.empty:
    print('No same-WHAT diagnostic pairs were retained.')
else:
    display(same_what.sort_values(['high_similarity', 'what_cosine_similarity', 'q_l1_distance'], ascending=False).head(30))
    high = same_what[same_what['high_similarity'] == True]
    print('High-similarity pairs:', len(high))
    if len(high):
        print('Fraction with different top local support:', high['different_top_support'].mean())
        print('Mean q L1 distance:', high['q_l1_distance'].mean())
        print('Mean evidence L1 distance:', high['evidence_l1_distance'].mean())

## Interpretation checklist

Strong support requires the evidence chain `ordered_gru > shared`, `ordered_gru > reset_gru`, `ordered_gru > clock`, and `ordered_gru > shuffled-q`, with q-only performance well below the full ordered model and same-WHAT pairs showing history-dependent q/evidence changes. `oracle_progress` is diagnostic: ordered beating it would support context beyond scalar progress; oracle beating ordered would indicate the expert mechanism works but history estimation remains limiting.